<a href="https://colab.research.google.com/github/Ahmet003-cod/Ben-kim/blob/main/BL%C4%B0P_MODEL_And_V%C4%B0T_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch pillow requests

In [ ]:
from transformers import BlipProcessor,BlipForConditionalGeneration
import requests
from PIL import Image
import torch

In [ ]:
processor=BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")#görseli  modele uygun girdin  tensorlerine cevirir
model=BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")#görsel girdisinden metin üretebilen  Blip modelein caption sürümü


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

In [ ]:
#Test Görseli İndir
img_url="https://i.pinimg.com/736x/50/47/7d/50477deba1cb0f8fbf84c4583e7c6e24.jpg"
image=Image.open(requests.get(img_url,stream=True).raw).convert("RGB")

#görseli tesöre çevir
inputs=processor(image,return_tensors="pt")#görseli normalize et ve pytorch tensörüne çevir "pt"=pytorch

#caption metin üretimi
with torch.no_grad():#yalnızca inference için
     output=model.generate(**inputs)#görselde çıktı üretme

#tokenleri insan okumasi için düzenle
caption=processor.decode(output[0],skip_special_tokens=True)
print(f"Üretilen Metin:{caption}")



Üretilen Metin:a group of people walking across a crosswalk


In [ ]:
import torch
from PIL import Image
import requests
from transformers import ViTImageProcessor, AutoTokenizer, VisionEncoderDecoderModel

# 1. Model, Processor ve Tokenizer Yükle
# Not: 'VisionEncoderDecoderModel' kullanmalısın, 'Config' sadece ayarları yükler.
model_name = "nlpconnect/vit-gpt2-image-captioning"

model = VisionEncoderDecoderModel.from_pretrained(model_name)
processor = ViTImageProcessor.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Cihaz Ayarı (GPU varsa kullanılır)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 3. Görseli Yükle (Örnek URL)
# Örnek kedi resmi
image = Image.open(requests.get(img_url, stream=True).raw)

# 4. Görseli Modele Uygun Hale Getir
pixel_values = processor(images=image, return_tensors="pt").pixel_values
pixel_values = pixel_values.to(device)

# 5. Modeli Çalıştıralım (Generate)
output_ids = model.generate(pixel_values, max_length=16, num_beams=4)

# 6. Sonuçları Çözümle ve Yazdır
caption = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(f"ViT-GPT2 Açıklaması: {caption.strip()}")

Loading weights:   0%|          | 0/445 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie decoder.transformer.wte.weight to decoder.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
VisionEncoderDecoderModel LOAD REPORT from: nlpconnect/vit-gpt2-image-captioning
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
decoder.transformer.h.{0...11}.attn.bias                  | UNEXPECTED |  | 
decoder.transformer.h.{0...11}.crossattention.masked_bias | UNEXPECTED |  | 
decoder.transformer.h.{0...11}.attn.masked_bias           | UNEXPECTED |  | 
decoder.transformer.h.{0...11}.crossattention.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ViT-GPT2 Açıklaması: people crossing a street at an intersection
